# Chapter 2 - Lab 5: Financial News Agent with Evaluator-Optimizer Pattern

This version keeps the **Evaluator-Optimizer** pattern, but fixes an important issue with hosted web search: direct source URLs are stored in response annotations / web-search source metadata and are not reliably preserved by `ItemHelpers.text_message_outputs()`. The notebook now extracts those URLs programmatically, following the current OpenAI Agents SDK web-search example.


## 1. Install dependencies


In [ ]:
!pip install -U openai openai-agents -q


After upgrading packages in Colab, restart the runtime if requested, then continue from the next cell.


## 2. Imports and API key


In [ ]:
from collections.abc import Mapping, Sequence
from dataclasses import dataclass
from datetime import datetime, timedelta
from typing import Any, Literal
from urllib.parse import urlsplit, urlunsplit

import os
from google.colab import userdata

from openai.types.responses.web_search_tool import Filters
from openai.types.shared.reasoning import Reasoning

from agents import (
    Agent,
    ItemHelpers,
    ModelSettings,
    Runner,
    TResponseInputItem,
    WebSearchTool,
)

OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY


## 3. Helpers to extract real web-search sources

The model output text alone is not enough to validate links. These helpers inspect the structured `new_items` returned by the Agents SDK and extract both inline URL citations and the URLs retrieved by the web-search call.


In [ ]:
@dataclass(frozen=True)
class URLCitation:
    title: str
    url: str


def get_field(obj: Any, key: str) -> Any:
    if isinstance(obj, Mapping):
        return obj.get(key)
    return getattr(obj, key, None)


def extract_url_citations(items: Sequence[Any]) -> list[URLCitation]:
    citations: list[URLCitation] = []
    seen: set[str] = set()

    for item in items:
        raw_item = get_field(item, "raw_item")
        if get_field(raw_item, "type") != "message":
            continue

        content = get_field(raw_item, "content")
        if not isinstance(content, list):
            continue

        for part in content:
            if get_field(part, "type") != "output_text":
                continue
            annotations = get_field(part, "annotations")
            if not isinstance(annotations, list):
                continue

            for annotation in annotations:
                if get_field(annotation, "type") != "url_citation":
                    continue
                url = get_field(annotation, "url")
                title = get_field(annotation, "title")
                if not isinstance(url, str) or url in seen:
                    continue
                seen.add(url)
                citations.append(
                    URLCitation(
                        title=title if isinstance(title, str) else url,
                        url=url,
                    )
                )

    return citations


def extract_web_search_source_urls(items: Sequence[Any]) -> list[str]:
    urls: list[str] = []
    seen: set[str] = set()

    for item in items:
        raw_item = get_field(item, "raw_item")
        if get_field(raw_item, "type") != "web_search_call":
            continue

        action = get_field(raw_item, "action")
        sources = get_field(action, "sources") if action else None
        if not isinstance(sources, list):
            continue

        for source in sources:
            url = get_field(source, "url")
            if not isinstance(url, str) or url in seen:
                continue
            seen.add(url)
            urls.append(url)

    return urls


def normalize_reuters_url(url: str) -> str | None:
    try:
        parsed = urlsplit(url)
    except ValueError:
        return None

    host = parsed.hostname.lower().rstrip(".") if parsed.hostname else ""
    if not (host == "reuters.com" or host.endswith(".reuters.com")):
        return None

    if parsed.scheme not in {"http", "https"}:
        return None

    path = parsed.path.rstrip("/")
    if not path:
        return None

    return urlunsplit(("https", host, path, "", ""))


def unique_reuters_urls(urls: Sequence[str]) -> list[str]:
    result: list[str] = []
    seen: set[str] = set()
    for url in urls:
        normalized = normalize_reuters_url(url)
        if normalized is None or normalized in seen:
            continue
        seen.add(normalized)
        result.append(normalized)
    return result


## 4. Searcher and evaluator agents

The searcher now asks for **inline citations** instead of forcing the language model to manually reproduce URLs. The direct Reuters URLs are extracted from the structured response afterwards.


In [ ]:
today_date = datetime.now().strftime("%Y-%m-%d")
two_days_ago = (datetime.now() - timedelta(days=2)).strftime("%Y-%m-%d")

ALLOWED_DOMAINS = ["reuters.com", "www.reuters.com"]

INSTRUCTIONS_NEWS_SEARCH = f"""
You are a financial news research agent.

Use web search to find genuine Reuters news articles relevant to the user's request.
The web-search tool is restricted to Reuters domains.

DATE WINDOW
- Earliest allowed publication date: {two_days_ago}
- Latest allowed publication date: {today_date}

SEARCH BEHAVIOR
- Search Reuters using several query formulations when necessary.
- Search topic synonyms, company names, sectors and subtopics rather than relying on one broad phrase.
- Do not substitute stock prices, ETF quotes, index values or generic market data for articles.
- Do not invent articles, dates, claims or sources.
- If the first query is insufficient, reformulate and search again.

OUTPUT
- Respect the exact number of news items requested by the user. If no number is specified, return 5.
- For every item include: headline, publication date, short summary and publisher (Reuters).
- IMPORTANT: include an inline web citation for every item.
- Do NOT worry about manually typing the full URL into the prose. The application extracts the real URL from the citation metadata.
- Only include articles whose publication date is inside the date window.
- If fewer valid articles are found after several searches, return the valid items found and say how many were found. Never fabricate missing items.
"""

web_news_searcher = Agent(
    name="web_news_searcher",
    model="gpt-5.6",
    instructions=INSTRUCTIONS_NEWS_SEARCH,
    tools=[
        WebSearchTool(
            filters=Filters(allowed_domains=ALLOWED_DOMAINS),
            search_context_size="high",
        )
    ],
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="low"),
        tool_choice="required",
        verbosity="low",
        response_include=["web_search_call.action.sources"],
    ),
)


@dataclass
class EvaluationFeedback:
    feedback: str
    score: Literal["successful", "unsuccessful"]


INSTRUCTIONS_NEWS_EVALUATOR = f"""
You are a strict evaluator of a Reuters financial-news search result.

You receive:
1. ORIGINAL USER REQUEST
2. NEWS SUMMARY
3. REUTERS URL CITATIONS extracted from the model response
4. REUTERS URLS RETRIEVED by the web-search tool

Evaluate only the requirements applicable to the original request.

A result is successful only if:
- it contains the requested number of news items (default 5 if unspecified);
- the items are relevant to the requested topic and region;
- each item has headline, publication date and short summary;
- each publication date is between {two_days_ago} and {today_date}, inclusive;
- the structured evidence contains enough genuine Reuters URLs to support the items;
- stock quotes, ETF quotes and generic market snapshots are not counted as news.

Do NOT require the URL to be manually embedded in NEWS SUMMARY when it is present in REUTERS URL CITATIONS / RETRIEVED URLS.
Do not invent missing requirements.

Return successful only if all applicable requirements are met. Otherwise return unsuccessful with specific actionable feedback.
"""

news_evaluator = Agent(
    name="news_evaluator",
    model="gpt-5.6",
    instructions=INSTRUCTIONS_NEWS_EVALUATOR,
    output_type=EvaluationFeedback,
    model_settings=ModelSettings(reasoning=Reasoning(effort="low")),
)


## 5. Evaluator-Optimizer loop with source diagnostics

The diagnostic block is deliberate. If the prose says that no articles were found but `RETRIEVED REUTERS URLS` contains Reuters links, then the problem is in synthesis/prompting rather than retrieval. If both lists are empty, the hosted search itself returned no Reuters sources.


In [ ]:
async def main() -> None:
    msg = input("User's request: " ).strip()

    max_iterations = 4
    latest_outline = ""
    latest_citations: list[URLCitation] = []
    latest_retrieved_urls: list[str] = []
    evaluator_feedback: str | None = None
    previous_reuters_urls: list[str] = []

    for iteration in range(1, max_iterations + 1):
        search_input: list[TResponseInputItem] = [
            {"content": msg, "role": "user"}
        ]

        if evaluator_feedback is not None:
            known_sources = "\n".join(f"- {url}" for url in previous_reuters_urls)
            refinement = (
                "The previous attempt failed evaluation. Perform a NEW web search from scratch.\n\n"
                f"Evaluator feedback:\n{evaluator_feedback}\n\n"
            )
            if known_sources:
                refinement += (
                    "The previous web-search call did retrieve these Reuters URLs. "
                    "Use them as leads, verify them, and search for additional qualifying Reuters articles if needed:\n"
                    f"{known_sources}\n\n"
                )
            refinement += "Correct every issue identified by the evaluator."
            search_input.append({"content": refinement, "role": "user"})

        news_searcher_result = await Runner.run(web_news_searcher, search_input)

        latest_outline = ItemHelpers.text_message_outputs(news_searcher_result.new_items)
        latest_citations = extract_url_citations(news_searcher_result.new_items)

        cited_reuters_urls = unique_reuters_urls([c.url for c in latest_citations])
        latest_retrieved_urls = unique_reuters_urls(
            extract_web_search_source_urls(news_searcher_result.new_items)
        )
        previous_reuters_urls = unique_reuters_urls(
            [*cited_reuters_urls, *latest_retrieved_urls]
        )

        print("\n\033[92m" + f"************************** NEWS SEARCH {iteration} **************************" + "\033[0m")
        print(latest_outline)

        print("\n\033[93m************************** WEB SEARCH DIAGNOSTICS **************************\033[0m")
        print(f"Reuters citations in final response: {len(cited_reuters_urls)}")
        for url in cited_reuters_urls:
            print(f"  CITED: {url}")
        print(f"Reuters URLs retrieved by web search: {len(latest_retrieved_urls)}")
        for url in latest_retrieved_urls:
            print(f"  RETRIEVED: {url}")

        citation_block = "\n".join(
            f"- {c.title}: {normalize_reuters_url(c.url)}"
            for c in latest_citations
            if normalize_reuters_url(c.url) is not None
        ) or "(none)"

        retrieved_block = "\n".join(
            f"- {url}" for url in latest_retrieved_urls
        ) or "(none)"

        evaluator_input = (
            f"ORIGINAL USER REQUEST:\n{msg}\n\n"
            f"NEWS SUMMARY:\n{latest_outline}\n\n"
            f"REUTERS URL CITATIONS:\n{citation_block}\n\n"
            f"REUTERS URLS RETRIEVED BY WEB SEARCH:\n{retrieved_block}"
        )

        print("\n\033[92m************************** RUNNING EVALUATION **************************\033[0m")
        news_evaluator_result = await Runner.run(news_evaluator, evaluator_input)
        result: EvaluationFeedback = news_evaluator_result.final_output

        print(f"\033[94mEvaluator score: {result.score}\033[0m")
        print(f"\033[94mEvaluator feedback: {result.feedback}\033[0m")

        if result.score == "successful":
            print("\033[92mEvaluation successful ==> stopping iteration.\033[0m")
            break

        evaluator_feedback = result.feedback

        if iteration == max_iterations:
            print("\033[91mReached max_iterations ==> stopping iteration.\033[0m")

    print("\n\033[92m************************** FINAL NEWS SET **************************\033[0m")
    print(latest_outline)

    final_urls = unique_reuters_urls(
        [*[c.url for c in latest_citations], *latest_retrieved_urls]
    )
    if final_urls:
        print("\nReuters URLs extracted from the structured web-search response:")
        for url in final_urls:
            print(f"- {url}")


## 6. Run

Suggested test:

`Give me the latest 5 Reuters articles from the last 2 days about OpenAI, Nvidia, Oracle, Adobe, AI infrastructure or AI investment in the United States.`


In [ ]:
await main()
